# Demo 02b — Blind (Derivative-Free) Optimization

**Class 02 · Block 3** (slides: `slides/02_optimization.md`)

The optimizer only sees **values** $f(x)$: no formula, no gradient. All algorithms come from
[pyBlindOpt](https://github.com/mariolpantunes/pyBlindOpt).

* **B1 — Local search:** random search, hill climbing and simulated annealing on the two 1D functions.
* **B2 — Initialization:** samplers (random, Latin hypercube, Sobol, chaotic) and strategies (OBL, QOBL, OBLESA).
* **B3 — Populations at work:** GA, DE and EGWO on a simple and a rugged 2D landscape.
* **B4 — A fair comparison:** many seeds, one evaluation budget, and a noisy objective.
* **B5 — Best of both worlds:** a blind global search polished by gradient descent.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pyBlindOpt as pbo
from pyBlindOpt import functions, init, utils

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
print("pyBlindOpt", pbo.__version__)

## Objectives and two small helpers

pyBlindOpt calls the objective with a whole population, a matrix of shape `(n_pop, d)`, when it can. Writing the
function with `x[..., i]` makes it work for a single point *and* for a population.

In [ ]:
def f_convex(x):
    return (x[..., 0] - 2.0) ** 2 + 1.0


def f_rugged(x):
    x = x[..., 0]
    return x**2 / 5 + np.sin(3 * x) + 0.3 * np.sin(11 * x)


BOUNDS_1D = np.array([[-4.0, 4.0]])
X_STAR_RUGGED = -0.6716


class Recorder:
    """A pyBlindOpt callback: keeps a copy of the population and scores at every epoch."""

    def __init__(self, optimizer=None):
        self.pops, self.scores = [], []
        if optimizer is not None:  # also keep the initial population (epoch 0)
            self(0, optimizer.scores, optimizer.pop)

    def __call__(self, epoch, scores, pop):
        self.pops.append(pop.copy())
        self.scores.append(scores.copy())


class Counted:
    """Wraps an objective and counts how many points were evaluated."""

    def __init__(self, f):
        self.f, self.n = f, 0

    def __call__(self, x):
        x = np.atleast_2d(x)
        self.n += x.shape[0]
        return self.f(x)


def run(cls, objective, bounds, **kwargs):
    """Build an optimizer, record every epoch, run it. Returns (best_x, best_f, recorder)."""
    opt = cls(objective, bounds, **kwargs)
    rec = Recorder(opt)
    opt.callbacks.append(rec)
    best_x, best_f = opt.optimize()
    return best_x, best_f, rec

## B1 — Random search, hill climbing, simulated annealing

* **Random search:** sample, keep the best. No memory of *where* good points were.
* **Hill climbing:** perturb the current point, $x' = x + \mathcal{N}(0,\sigma^2)$; move only if $f(x') < f(x)$.
* **Simulated annealing:** also accept a *worse* point with probability $e^{-\Delta f / T}$; the temperature
  $T_t = T_0/(t+1)$ cools down, so it explores first and exploits later.

All three start from $x_0 = 3$ and get the same budget: 200 evaluations.

In [ ]:
start = np.array([[3.0]])
common = dict(n_iter=200, seed=7)
runs = {
    "Random search": run(pbo.RandomSearch, f_rugged, BOUNDS_1D, n_pop=1, **common),
    "Hill climbing": run(pbo.HillClimbing, f_rugged, BOUNDS_1D, population=start, step_size=0.5, **common),
    "Simulated annealing": run(pbo.SimulatedAnnealing, f_rugged, BOUNDS_1D, population=start, step_size=0.5, temp=30.0, **common),
}

grid = np.linspace(-4, 4, 1001)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(grid[:, 0], f_rugged(grid), color="k", lw=1)
for (name, (bx, bf, rec)), color in zip(runs.items(), ["tab:gray", "tab:red", "tab:green"]):
    xs = np.concatenate(rec.pops)[:, 0]
    axes[0].plot(xs, f_rugged(xs[:, None]), ".", color=color, ms=4, alpha=0.5)
    axes[0].plot(bx[0], bf, "o", color=color, ms=10, mec="k", label=f"{name}: best x={bx[0]:+.2f}, f={bf:+.2f}")
    axes[1].plot(xs, color=color, lw=1, label=name)
axes[0].set(xlabel="x", ylabel="f(x)", title="visited points and best found")
axes[0].legend(fontsize=8)
axes[1].axhline(X_STAR_RUGGED, color="k", ls="--", lw=1, label="global minimum")
axes[1].set(xlabel="iteration", ylabel="current x", title="the current point over time")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

One run is an anecdote. Over **100 random starting points**, how often does each method end in the global basin?

In [ ]:
def success_rate(cls, n_runs=100, **kwargs):
    hits = 0
    for seed in range(n_runs):
        x0 = np.random.default_rng(1000 + seed).uniform(-4, 4, (1, 1))
        pop = None if cls is pbo.RandomSearch else x0
        bx, _ = cls(f_rugged, BOUNDS_1D, population=pop, n_iter=200, seed=seed, **kwargs).optimize()
        hits += abs(bx[0] - X_STAR_RUGGED) < 0.05
    return hits / n_runs


for name, cls, kw in [
    ("Random search", pbo.RandomSearch, dict(n_pop=1)),
    ("Hill climbing, sigma=0.1", pbo.HillClimbing, dict(step_size=0.1)),
    ("Hill climbing, sigma=0.5", pbo.HillClimbing, dict(step_size=0.5)),
    ("Simulated annealing, sigma=0.5", pbo.SimulatedAnnealing, dict(step_size=0.5, temp=30.0)),
    ("Hill climbing, sigma=1.0", pbo.HillClimbing, dict(step_size=1.0)),
]:
    print(f"{name:32s} reaches the global basin in {success_rate(cls, **kw):4.0%} of 100 runs")

**On the convex function** every method works, but compare the cost: gradient descent needed ~25 *gradient* steps
(notebook 02a); a blind method pays in *function evaluations*.

In [ ]:
for name, cls, kw in [
    ("Random search", pbo.RandomSearch, dict(n_pop=1)),
    ("Hill climbing", pbo.HillClimbing, dict(population=start, step_size=0.5)),
    ("Simulated annealing", pbo.SimulatedAnnealing, dict(population=start, step_size=0.5, temp=30.0)),
]:
    f = Counted(f_convex)
    bx, bf = cls(f, BOUNDS_1D, n_iter=200, seed=7, **kw).optimize()
    print(f"{name:20s} x = {bx[0]:.4f} (x* = 2), f - f* = {bf - 1:.1e}, evaluations = {f.n}")

**Takeaway:** a small step exploits (precise but trapped), a large step explores (escapes but imprecise).
Annealing moves from one to the other over time. Note that plain random search is excellent **in 1D**: 200 samples
cover a line densely. In 10 dimensions (B4) it is the worst method. The price of not having a gradient is paid in
evaluations.

## B2 — Initialization: where do we start?

Population methods start from $N$ points. How they are spread matters most when the budget is small.

**Samplers** (where the $N$ points go): uniform random, Latin hypercube (one point per row and column strip),
Sobol (a low-discrepancy sequence), chaotic (a logistic map).

In [ ]:
BOUNDS_2D = np.array([[-5.0, 5.0]] * 2)
N = 32
samplers = {
    "Random": utils.RandomSampler,
    "Latin hypercube": utils.HLCSampler,
    "Sobol": utils.SobolSampler,
    "Chaotic": utils.ChaoticSampler,
}


def min_distance(p):
    d = np.linalg.norm(p[:, None] - p[None, :], axis=-1)
    return d[np.triu_indices(len(p), 1)].min()


fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (name, S) in zip(axes, samplers.items()):
    pop = init.get_initial_population(N, BOUNDS_2D, S(np.random.default_rng(3)))
    ax.scatter(*pop.T, s=14)
    for v in np.linspace(-5, 5, 9):
        ax.axvline(v, color="0.9", lw=0.5, zorder=0)
        ax.axhline(v, color="0.9", lw=0.5, zorder=0)
    ax.set(xlim=(-5, 5), ylim=(-5, 5), aspect="equal", title=f"{name}\nclosest pair: {min_distance(pop):.2f}")
plt.tight_layout()
plt.show()

**Strategies** use the objective to *choose* the starting points:

* **OBL** (opposition-based learning): for every sample $x$ also evaluate its opposite $\breve x = l + u - x$ and keep
  the best $N$ of the $2N$.
* **QOBL** (quasi-opposition): the opposite is drawn between the centre of the box and $\breve x$.
* **OBLESA**: OBL plus probes placed in the *empty space* that neither block covered (EmptySpaceSearch), then the
  best $N$ with a diversity bonus.

Each strategy spends extra evaluations up front ($2N$ for OBL, $3N$ for OBLESA).

In [ ]:
def shifted(f, shift):
    """Move the optimum of f from the origin to `shift` (as the BBOB benchmarks do)."""

    def g(x):
        return f(x - shift)

    return g


SHIFT_2D = np.array([1.3, -2.2])
rastrigin = shifted(functions.rastrigin, SHIFT_2D)
X, Y = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-5, 5, 300))
Z = rastrigin(np.stack([X.ravel(), Y.ravel()], axis=1)).reshape(X.shape)

strategies = {
    "Random (no strategy)": lambda rng: init.get_initial_population(N, BOUNDS_2D, utils.RandomSampler(rng)),
    "OBL": lambda rng: init.opposition_based(rastrigin, BOUNDS_2D, population=utils.RandomSampler(rng), n_pop=N, seed=rng),
    "QOBL": lambda rng: init.quasi_opposition_based(rastrigin, BOUNDS_2D, population=utils.RandomSampler(rng), n_pop=N, seed=rng),
    "OBLESA": lambda rng: init.oblesa(rastrigin, BOUNDS_2D, population=utils.RandomSampler(rng), n_pop=N, seed=rng),
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (name, make) in zip(axes, strategies.items()):
    pop = make(np.random.default_rng(3))
    ax.contourf(X, Y, Z, levels=30, cmap="viridis")
    ax.scatter(*pop.T, s=16, color="w", edgecolor="k")
    ax.plot(*SHIFT_2D, "r*", ms=14)
    ax.set(aspect="equal", title=f"{name}\nmean f = {rastrigin(pop).mean():.1f}")
plt.tight_layout()
plt.show()

**A benchmark pitfall.** The textbook functions have their optimum at the **centre** of a symmetric box. There,
OBL is useless ($f(-x) = f(x)$: the opposite point has the same value) and QOBL, which samples towards the centre,
looks brilliant. Real problems have no reason to be centred, so from here on every function is **shifted**:
$f(x - s)$ with a random $s$.

**Does it matter?** Differential evolution on a shifted 10-dimensional Rastrigin function with a *small* budget
(20 individuals, 40 generations), 30 seeds per initializer. Lower is better.

In [ ]:
D = 10
BOUNDS_10D = np.array([[-5.0, 5.0]] * D)
SHIFT_10D = np.random.default_rng(2026).uniform(-3, 3, D)
rastrigin_10d = shifted(functions.rastrigin, SHIFT_10D)


def make_population(kind, rng, n_pop=20):
    sampler_cls = {"Random": utils.RandomSampler, "LHS": utils.HLCSampler, "Sobol": utils.SobolSampler}.get(kind)
    if sampler_cls is not None:
        return init.get_initial_population(n_pop, BOUNDS_10D, sampler_cls(rng))
    base = utils.RandomSampler(rng)
    if kind == "OBL":
        return init.opposition_based(rastrigin_10d, BOUNDS_10D, population=base, n_pop=n_pop, seed=rng)
    if kind == "QOBL":
        return init.quasi_opposition_based(rastrigin_10d, BOUNDS_10D, population=base, n_pop=n_pop, seed=rng)
    return init.oblesa(rastrigin_10d, BOUNDS_10D, population=base, n_pop=n_pop, seed=rng)


rows = []
t0 = time.perf_counter()
for kind in ["Random", "LHS", "Sobol", "OBL", "QOBL", "OBLESA"]:
    for seed in range(30):
        rng = np.random.default_rng(seed)
        pop = make_population(kind, rng)
        _, best = pbo.differential_evolution(rastrigin_10d, BOUNDS_10D, population=pop, n_iter=40, seed=rng)
        rows.append({"init": kind, "seed": seed, "initial best": float(rastrigin_10d(pop).min()), "final best": float(best)})
results = pl.DataFrame(rows)
print(f"{len(rows)} runs in {time.perf_counter() - t0:.1f} s")
summary = results.group_by("init", maintain_order=True).agg(
    pl.col("initial best").median().alias("median initial best"),
    pl.col("final best").median().alias("median final best"),
    pl.col("final best").quantile(0.25).alias("q25"),
    pl.col("final best").quantile(0.75).alias("q75"),
)
print(summary)

fig, ax = plt.subplots()
kinds = summary["init"].to_list()
ax.boxplot([results.filter(pl.col("init") == k)["final best"].to_numpy() for k in kinds], tick_labels=kinds)
ax.set(ylabel="best f after 40 generations", title="DE on Rastrigin-10D: effect of the initial population (30 seeds)")
plt.show()

**Takeaway:** a better spread (LHS, Sobol) makes the *coverage* more even, but with 20 points in 10 dimensions
it barely changes the result. Spending evaluations to *select* the start (QOBL, OBLESA) gives DE a head start that
survives 40 generations. The effect is largest when the budget is small, varies across seeds (look at the boxes,
not only the medians), and depends on the algorithm: pyBlindOpt's documentation notes that EGWO hardly responds
to a better starting population.

## B3 — Populations at work: GA, DE and EGWO

* **GA:** select parents (tournament), recombine them (blend crossover), mutate the children, keep a few elites.
* **DE:** a mutant $v = x_{best} + F\,(x_{r_1} - x_{r_2})$ built from *difference vectors* of the population,
  crossed with the parent, kept only if it is better.
* **EGWO:** the three best wolves ($\alpha,\beta,\delta$) estimate the prey; every wolf moves around that estimate,
  with a radius $a$ that shrinks from 2 to 0.

Snapshots of the population on a simple bowl (Sphere) and on a rugged landscape (Rastrigin).

In [ ]:
algorithms = {
    "GA": (pbo.GeneticAlgorithm, {}),
    "DE": (pbo.DifferentialEvolution, {}),
    "EGWO": (pbo.EGWO, {}),
}
snapshots = [0, 3, 10, 30]


def snapshot_grid(objective, title):
    Zf = objective(np.stack([X.ravel(), Y.ravel()], axis=1)).reshape(X.shape)
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for row, (name, (cls, kw)) in zip(axes, algorithms.items()):
        _, best, rec = run(cls, objective, BOUNDS_2D, n_pop=20, n_iter=30, seed=11, **kw)
        for ax, epoch in zip(row, snapshots):
            ax.contourf(X, Y, np.log1p(Zf - Zf.min()), levels=25, cmap="viridis")
            ax.scatter(*rec.pops[epoch].T, s=14, color="w", edgecolor="k")
            ax.plot(*SHIFT_2D, "r*", ms=12)
            ax.set(xticks=[], yticks=[], aspect="equal")
            ax.set_title(f"{name}, epoch {epoch}: best {rec.scores[epoch].min():.2g}", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


snapshot_grid(shifted(functions.sphere, SHIFT_2D), "Sphere: one smooth basin")
snapshot_grid(rastrigin, "Rastrigin: a grid of local minima")

## B4 — A fair comparison

Stochastic optimizers must be compared like models in Class 01: **many seeds, the same budget, a spread and not
just a single number.** Budget here: 20 individuals × 100 epochs (≈ 2000 evaluations), 10 dimensions, 15 seeds.

The last column is a **noisy** Sphere, $f(x) + \varepsilon$ with $\varepsilon \sim \mathcal N(0, 0.5^2)$: every
evaluation of the same point returns a different value, like the loss of a model trained with a random seed.
We report the *true* $f$ at the returned point, because the best *observed* value is optimistically biased.

In [ ]:
NOISE = np.random.default_rng(123)


sphere_10d = shifted(functions.sphere, SHIFT_10D)
ackley_10d = shifted(functions.ackley, SHIFT_10D)


def noisy_sphere(x):
    return sphere_10d(x) + NOISE.normal(0, 0.5, np.shape(x)[:-1])


problems = {  # name: (what the optimizer sees, the true function)
    "sphere": (sphere_10d, sphere_10d),
    "rastrigin": (rastrigin_10d, rastrigin_10d),
    "ackley": (ackley_10d, ackley_10d),
    "noisy sphere": (noisy_sphere, sphere_10d),
}
contenders = {
    "Random search": (pbo.RandomSearch, {}),
    "Hill climbing": (pbo.HillClimbing, dict(step_size=0.1)),
    "Sim. annealing": (pbo.SimulatedAnnealing, dict(step_size=0.1, temp=5.0)),
    "GA": (pbo.GeneticAlgorithm, {}),
    "DE": (pbo.DifferentialEvolution, {}),
    "DE (SHADE)": (pbo.DifferentialEvolution, dict(variant="current-to-pbest/1/bin", policy="shade")),
    "EGWO": (pbo.EGWO, {}),
}

rows = []
t0 = time.perf_counter()
for pname, (objective, truth) in problems.items():
    for aname, (cls, kw) in contenders.items():
        for seed in range(15):
            bx, _ = cls(objective, BOUNDS_10D, n_pop=20, n_iter=100, seed=seed, **kw).optimize()
            rows.append({"problem": pname, "algorithm": aname, "seed": seed, "f": float(truth(bx[None, :])[0])})
bench = pl.DataFrame(rows)
print(f"{len(rows)} runs in {time.perf_counter() - t0:.1f} s")
table = bench.group_by("algorithm", "problem", maintain_order=True).agg(pl.col("f").median()).pivot(
    on="problem", index="algorithm", values="f"
)
with pl.Config(float_precision=3, tbl_rows=20):
    print("median true f over 15 seeds (lower is better)")
    print(table)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
names = list(contenders)
for ax, pname in zip(axes, problems):
    data = [bench.filter((pl.col("problem") == pname) & (pl.col("algorithm") == a))["f"].to_numpy() for a in names]
    ax.boxplot([np.maximum(d, 1e-12) for d in data], tick_labels=names)
    ax.set_yscale("log")
    ax.set_title(pname)
    ax.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("true f of the returned point (log)")
plt.tight_layout()
plt.show()

**Takeaway:**

* In 10 dimensions, single-point local search (hill climbing, annealing) and random search are far behind every
  population method: the curse of dimensionality from Class 01.
* Among population methods the ranking changes with the landscape (No Free Lunch). Classic DE (`best/1`) is greedy:
  excellent on some seeds, stuck on others (tall boxes). The adaptive variant (SHADE) is the most consistent.
* Noise hurts the greedy methods most: a lucky evaluation is taken for a real improvement.

## B5 — Best of both worlds: global blind search, local gradient polish

When the gradient exists, use it where it is strong (local refinement), and let a population handle the global
part. DE gets a small budget on the 1D rugged function, then gradient descent (JAX) polishes its answer.

In [ ]:
def f_rugged_jax(x):
    return x**2 / 5 + jnp.sin(3 * x) + 0.3 * jnp.sin(11 * x)


df = jax.jit(jax.grad(f_rugged_jax))
for seed in range(5):
    counted = Counted(f_rugged)
    bx, bf = pbo.differential_evolution(counted, BOUNDS_1D, n_pop=8, n_iter=5, seed=seed)
    x = bx[0]
    for _ in range(50):
        x = x - 0.02 * float(df(x))
    print(f"seed {seed}: DE ({counted.n} evaluations) x = {bx[0]:+.4f}, f = {bf:+.5f}  ->  "
          f"+50 GD steps x = {x:+.4f}, f = {float(f_rugged_jax(x)):+.5f}")

**Takeaway:** the blind stage decides *which basin* (seeds 0 and 4 picked the second-best one with only 48
evaluations); the gradient stage then reaches the bottom of that basin in a few cheap steps. Such hybrids are called
*memetic* algorithms.

## See it live

* **pyOptViewer** (`github.com/mariolpantunes/pyOptViewer`): every pyBlindOpt algorithm, sampler and strategy on the
  2D benchmark functions, animated epoch by epoch, with the explored surface the algorithm "sees".
  Run `python -m optviewer` and open `http://127.0.0.1:8000`.
* **blindgame**: *be* the optimizer. Hidden 2D functions, 5 evaluations each, a class leaderboard, in the spirit of
  the GECCO 2025 Fun Competition. Afterwards, the same budget is given to the algorithms above.